# Python 异步编程详解（asyncio 教学级 Notebook）

> **一句话本质**：异步 = **单线程内的协作式调度** —— 遇到 I/O 就把当前协程挂起，让事件循环去跑其他任务；它适合**高并发 I/O**，不适合 CPU 密集。

## 学习目标

1. 理解同步阻塞、多线程、异步三种并发模型的本质差异与选型依据
2. 掌握事件循环的工作原理，以及 `asyncio.run()` / 顶层 `await` 两种驱动方式
3. 熟练使用 `async` / `await` 语法，分清协程、Task、Future 三类可等待对象
4. 掌握 `gather` / `create_task` / `wait` / `as_completed` / `wait_for` / `shield` / `TaskGroup` 等核心 API
5. 掌握信号量限流、生产者消费者、超时重试、单飞缓存、优雅关闭等工程模式
6. 能识别并修复 12 类常见异步陷阱

## 业务场景说明

本 Notebook 的示例统一围绕 **LLMOps API 服务** 业务场景取材：**高并发调用 LLM 接口、批量评测任务、限流与重试、异步缓存、流式输出**。所有"网络调用"均用**确定性的模拟函数**（固定延迟、按编号确定性失败）代替真实网络，保证每次执行输出完全一致、可复现。真实第三方库（aiohttp / httpx / asyncpg 等）的写法在第 9 章以代码块形式给出。

## 环境与阅读说明

| 事项 | 说明 |
|---|---|
| Python 版本 | 3.13（语法特性标注了最低版本要求，如 TaskGroup 需 3.11+） |
| 运行环境 | Jupyter 内核本身运行在事件循环里，因此 Notebook 内使用**顶层 `await`** 驱动协程 |
| 脚本写法 | 在 `.py` 脚本中，等价写法是 `asyncio.run(main())`（详见 0.4 节） |
| 代码风格 | 注释采用「编号知识点 + 逐行中文注释」；所有代码单元均已真实执行、0 报错 |
| 修正说明 | 源文档中 3 处技术表述偏差已修正，均在正文以 **【修正说明】** 标注并附验证输出（汇总见 12.6） |


## 0 前置知识

**本章解决什么问题**：异步编程建立在 4 个语言基础特性之上 —— 函数调用语义、生成器的"暂停/恢复"、上下文管理器、事件循环环境。先把这 4 块地基打牢，后面 `async`/`await` 的每一条规则都能落到实处。每个特性都配一个可运行的验证示例。

### 0.1 函数调用即执行 —— 与"协程调用不执行"对照


In [1]:
# [前置1] 普通函数：调用立即执行函数体，直接返回结果
def sync_fetch(url):
    # ① 一调用就进入函数体
    # ② 立即返回结果（这里是同步构造字符串，无任何等待）
    return f"data from {url}"

result = sync_fetch("url-1")            # 调用即执行
print(type(result), "→", result)        # <class 'str'>，拿到的是【结果本身】


<class 'str'> → data from url-1


In [2]:
# [前置2] 协程函数：调用【不执行】函数体，只返回一个"协程对象"
async def async_fetch(url):
    # ① 注意：这一行 print 只有被 await 驱动时才会执行
    print(f"  （函数体开始执行：{url}）")
    return f"data from {url}"

c = async_fetch("url-1")    # ① 只创建了协程对象，函数体一行都没跑（上面没有输出）
print(type(c))              # ② <class 'coroutine'>，拿到的是【待执行的执行体】
result = await c            # ③ await 驱动执行（Jupyter 支持顶层 await）
print(result)


<class 'coroutine'>
  （函数体开始执行：url-1）
data from url-1


### 0.2 生成器：能"暂停 / 恢复"的函数（协程机制的祖先）

协程"挂起后还能从断点恢复"的能力，直接继承自生成器的 `yield` 语义：

- `yield` = 暂停点：把中间结果交出去，函数冻结在原地
- `next()` / `send()` = 恢复点：从上次冻结的位置继续执行

`async`/`await` 本质上是这套机制在事件循环上的升级版：`await` ≈ "在 yield 的基础上，还告诉事件循环'我等的东西好了再叫我'"。


In [3]:
# [前置3] 生成器：yield 让函数可以暂停/恢复 —— 这是协程"挂起"机制的祖先
def gen():
    print("① 第一次被驱动：执行到 yield 暂停")
    x = yield 1                     # ① 暂停在 yield，把 1 交出去；恢复时收到 send 传入的值
    print(f"② 收到 send 传入的 {x}，从断点继续执行")
    yield x * 10                    # ② 再次暂停，交出 x*10

g = gen()                           # 注意：调用 gen() 也不执行函数体（和协程一样！）
print("next →", next(g))            # 驱动到第一个 yield，得到 1
print("send →", g.send(5))          # 恢复执行，得到 50


① 第一次被驱动：执行到 yield 暂停
next → 1
② 收到 send 传入的 5，从断点继续执行
send → 50


### 0.3 上下文管理器 `with`（为 `async with` 铺垫）

`with` 的价值：**无论中间是否抛异常，退出时一定执行清理**。异步版本 `async with` 的语义完全相同，只是进入/退出这两个动作本身可能是 I/O 操作（如建立/断开网络连接）。


In [4]:
# [前置4] 同步 with：进入/退出自动调用 __enter__ / __exit__
class SyncResource:
    def __enter__(self):
        print("① 打开资源")
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("② 关闭资源（即使中途异常也会执行）")
        return False                # False = 不吞异常，继续向外抛

with SyncResource() as r:
    print("   使用资源中")


① 打开资源
   使用资源中
② 关闭资源（即使中途异常也会执行）


### 0.4 Jupyter 与事件循环（重要的环境前提）

**一个线程同一时刻只能有一个运行中的事件循环。** Jupyter（IPython）内核本身就把事件循环跑起来了，因此：

| 环境 | 驱动异步代码的方式 |
|---|---|
| `.py` 脚本 | `asyncio.run(main())`（入口处调用一次） |
| Jupyter Notebook | 直接写顶层 `await main()`（内核的事件循环帮你驱动） |
| Jupyter 里调 `asyncio.run()` | ❌ `RuntimeError`：不能在已运行的事件循环里再启动一个 |


In [5]:
# [前置5] 验证：Notebook 里直接调用 asyncio.run 会发生什么
import asyncio

async def hello():
    return "hello"

try:
    c = hello()
    asyncio.run(c)                    # ① 反例：内核已有一个运行中的事件循环
except RuntimeError as e:
    print("① 报错 →", e)
    c.close()                         # 关闭未被驱动的协程，避免 "never awaited" 警告

print("② 正确：顶层 await →", await hello())   # ② 内核事件循环直接驱动


① 报错 → asyncio.run() cannot be called from a running event loop
② 正确：顶层 await → hello


> **小结**：本 Notebook 所有异步单元用顶层 `await` 驱动；把代码搬到 `.py` 文件时，只需把顶层 `await xxx` 包进 `async def main()` 再 `asyncio.run(main())` 即可。

---

## 1 为什么需要异步

**本章解决什么问题**：用同一个"批量调用 LLM API"任务，分别跑同步、多线程、异步三种实现并计时对比，直观回答"异步到底省在哪里、什么时候不该用"。

### 1.1 同步阻塞的问题


In [6]:
# [1-1] 同步串行：每个请求都要干等上一步完成
import time

def fetch(url):
    time.sleep(0.3)                  # ① 模拟网络请求：同步阻塞，线程干等
    return f"data from {url}"

t0 = time.perf_counter()
for i in range(3):
    fetch(f"url{i}")
t_serial = time.perf_counter() - t0

print(f"串行总耗时 ≈ 0.9s（3 × 0.3）")
print("验证：0.8 < 耗时 < 1.2 →", 0.8 < t_serial < 1.2)


串行总耗时 ≈ 0.9s（3 × 0.3）
验证：0.8 < 耗时 < 1.2 → True


### 1.2 多线程方案

把 `fetch` 丢进线程池，三个请求各自在自己的线程里 `sleep`，互不等待。**可以并发，但代价是**：线程有创建/切换开销；GIL 使 CPU 密集任务无法真正并行；共享状态需要加锁。


In [7]:
# [1-2] 多线程：线程池并发执行三个阻塞请求
from concurrent.futures import ThreadPoolExecutor

def fetch(url):
    time.sleep(0.3)
    return f"data from {url}"

t0 = time.perf_counter()
with ThreadPoolExecutor(3) as pool:
    results = list(pool.map(fetch, ["url1", "url2", "url3"]))
t_thread = time.perf_counter() - t0

print("结果:", results)
print("验证：耗时 < 0.45（近似 3 路并行）→", t_thread < 0.45)


结果: ['data from url1', 'data from url2', 'data from url3']
验证：耗时 < 0.45（近似 3 路并行）→ True


### 1.3 异步方案

单线程内并发：`await asyncio.sleep()` 时当前协程**挂起**（而不是线程干等），事件循环趁机去推进其他任务。**等待时间重叠了，总耗时 ≈ 最长的那一次。**


In [8]:
# [1-3] 异步：单线程内，等待 I/O 时切换任务
import asyncio

async def fetch(url):
    await asyncio.sleep(0.3)         # ① 非阻塞等待：让出控制权，事件循环去跑别的任务
    return f"data from {url}"

async def main():
    return await asyncio.gather(     # ② 三个请求并发推进
        fetch("url1"), fetch("url2"), fetch("url3")
    )

t0 = time.perf_counter()
results = await main()
t_async = time.perf_counter() - t0

print("结果:", results)
print("验证：耗时 < 0.45（≈ 最长一次 0.3s）→", t_async < 0.45)


结果: ['data from url1', 'data from url2', 'data from url3']
验证：耗时 < 0.45（≈ 最长一次 0.3s）→ True


### 1.4 三种并发模型对比

| 模型 | 适用场景 | 开销 | 共享状态 |
|---|---|---|---|
| 多进程 | CPU 密集型 | 大 | 需要 IPC |
| 多线程 | I/O 密集型（中低并发） | 中 | 需要锁 |
| **异步** | **I/O 密集型、高并发** | **小** | **单线程无锁** |

> 结论：三个请求耗时 **0.9s（串行）→ ~0.3s（多线程/异步）**。异步用"挂起协程"代替"占着线程干等"，在高并发（成百上千路 I/O）下开销优势最明显。

---

## 2 事件循环

**本章解决什么问题**：事件循环是异步的心脏。本章用输出验证它的调度顺序，讲清 `asyncio.run()` 做了什么、什么时候需要手动管理循环，以及"一个线程一个循环"这条铁律。

事件循环维护一个就绪任务队列，不断循环：

1. 执行就绪的任务
2. 遇到 `await`：挂起当前任务，注册"就绪后唤醒我"的回调
3. I/O 就绪 / 定时器到点：恢复对应任务
4. 重复，直到所有任务完成


In [9]:
# [2-1] 观察事件循环的调度顺序：谁先挂起、谁先恢复
import asyncio, time

async def task(name, delay):
    print(f"START {name}（将让出 {delay}s）")
    await asyncio.sleep(delay)       # 挂起点：此刻事件循环切换去推进其他任务
    print(f"RESUME {name}（睡了 {delay}s 后被唤醒）")

t0 = time.perf_counter()
await asyncio.gather(
    task("T1", 0.3),
    task("T2", 0.1),
    task("T3", 0.2),
)
t = time.perf_counter() - t0

# 三个任务几乎同时 START；但 RESUME 顺序 = 睡眠时长顺序（T2→T3→T1）
print("验证：总耗时 ≈ 最长延迟 0.3s（< 0.45）→", t < 0.45)


START T1（将让出 0.3s）
START T2（将让出 0.1s）
START T3（将让出 0.2s）
RESUME T2（睡了 0.1s 后被唤醒）


RESUME T3（睡了 0.2s 后被唤醒）
RESUME T1（睡了 0.3s 后被唤醒）
验证：总耗时 ≈ 最长延迟 0.3s（< 0.45）→ True


### 2.1 `asyncio.run()` 与手动管理事件循环

`asyncio.run(main())`（脚本入口）会依次做三件事：

1. 创建一个**新**事件循环
2. 在该循环中运行 `main()` 直到完成
3. 关闭事件循环

手动管理（`new_event_loop` + `run_until_complete`）一般不推荐，仅在嵌入式场景（如在无循环的线程里起一个独立循环）使用。下面的示例在**子线程**中演示手动管理（子线程没有运行中的循环，所以合法且可运行）：


In [10]:
# [2-2] 手动管理事件循环：在"没有运行中循环"的子线程里创建并使用
import asyncio, threading

async def hello():
    return "hello from manual loop"

def run_in_thread():
    # ① 子线程当前没有事件循环 → 手动创建一个
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(hello())   # ② 在这个循环里跑到协程结束
    finally:
        loop.close()                              # ③ 用完关闭
        asyncio.set_event_loop(None)

# 主线程（Notebook 内核）的事件循环把阻塞函数丢进线程池执行
result = await asyncio.get_running_loop().run_in_executor(None, run_in_thread)
print("子线程手动循环的运行结果 →", result)


子线程手动循环的运行结果 → hello from manual loop


### 2.2 反例剧场

**易错点：以为任何地方都能 `get_running_loop()` / 操作循环。** 铁律是"一个线程同一时刻只有一个运行中的循环"——在从未启动循环的线程里调用会直接报错。


In [11]:
# [反例2-1] 在没有事件循环的线程里调用 get_running_loop
from concurrent.futures import ThreadPoolExecutor

def no_loop_here():
    try:
        asyncio.get_running_loop()    # ① 该线程没有"正在运行中"的循环
    except RuntimeError as e:
        return f"报错 → {e}"

with ThreadPoolExecutor(1) as pool:
    print("子线程中 →", pool.submit(no_loop_here).result())

# 对比：Notebook 主线程有内核的事件循环，可以正常拿到
print("主线程中 → 拿到运行中的循环:", asyncio.get_running_loop() is not None)


子线程中 → 报错 → no running event loop
主线程中 → 拿到运行中的循环: True


---

## 3 `async` / `await` 语法

**本章解决什么问题**：吃透"调用协程函数 ≠ 执行协程"、"await 到底做了什么"、以及四类可等待对象 —— 这是后面所有 API 的语法地基。

### 3.1 定义协程

`async def` 定义的函数叫**协程函数**；调用它返回**协程对象**，函数体不会执行。驱动一个协程对象的三种方式：

1. `await` 它（当前协程挂起，等它跑完）
2. `asyncio.create_task()` 包装成 Task 交给事件循环调度
3. `asyncio.run()` / 事件循环直接运行（脚本入口）


In [12]:
# [3-1] 协程对象与其驱动方式
async def fetch(url):
    await asyncio.sleep(0.05)         # 模拟一次 I/O
    return f"data from {url}"

# 驱动方式①：await —— 当前执行流挂起，等协程跑完拿结果
r1 = await fetch("u1")
print("await 驱动 →", r1)

# 驱动方式②：create_task —— 注册进事件循环，后台调度（见 4.4 详解）
t = asyncio.create_task(fetch("u2"))
r2 = await t                          # await Task 只是等它的结果
print("create_task 驱动 →", r2)


await 驱动 → data from u1


create_task 驱动 → data from u2


### 3.2 / 3.3 协程 vs 普通函数

| 对比项 | 普通函数 `def` | 协程函数 `async def` |
|---|---|---|
| 调用后 | 立即执行函数体 | 只返回协程对象，**不执行** |
| 返回值 | 函数的返回值 | `<coroutine object>` |
| 怎么拿到结果 | 直接用 | 必须 `await` 或交给事件循环 |
| `await` 能否出现在里面 | ❌（SyntaxError） | ✅ |

`await` 只能出现在 `async def` 函数内部（Jupyter 顶层 await 是内核帮你包了一层 async 上下文）。


In [13]:
# [3-2] 协程与普通函数的调用结果对照
def normal():
    return 1

async def coro():
    return 2

print("normal() →", normal())         # ① 直接是返回值
c = coro()
print("coro()    →", type(c), "（repr 形如 <coroutine object coro at 0x...>）")
print("await coro() →", await c)      # ② 驱动后才拿到返回值
# 验证 await 的执行顺序：先执行 await 右侧
async def side_effect():
    print("  （await 右侧的协程先执行）")
    return "R"

async def main():
    print("await 之前")
    r = await side_effect()           # ① 先执行右侧 side_effect
    print("await 之后，拿到:", r)      # ② 再继续本协程
await main()


normal() → 1
coro()    → <class 'coroutine'> （repr 形如 <coroutine object coro at 0x...>）
await coro() → 2
await 之前
  （await 右侧的协程先执行）
await 之后，拿到: R


### 3.4 `await` 的语义与可等待对象

`await x` 会：

1. 执行右侧的可等待对象
2. 如果它需要等待 I/O：挂起当前协程（控制权交还事件循环）
3. 事件循环切换去执行其他就绪任务
4. 等待完成后恢复当前协程，拿到结果

**可等待对象（Awaitable）共四类：**

| 类型 | 说明 | 本节验证 |
|---|---|---|
| 协程对象 | `async def` 调用的产物 | 3-1 ✅ |
| `asyncio.Task` | 被事件循环调度的协程 | 3-3 ✅ |
| `asyncio.Future` | 底层结果占位符（"将来才有值"） | 3-3 ✅ |
| 实现了 `__await__` 的对象 | 自定义可等待对象 | 3-4 ✅ |


In [14]:
# [3-3] Task 与 Future 都是可等待对象
async def coro():
    await asyncio.sleep(0.05)
    return "coro result"

t = asyncio.create_task(coro())                 # ① Task：被调度的协程
print("await Task →", await t)

fut = asyncio.get_running_loop().create_future()
fut.set_result("future result")                 # ② Future：占位符，由"别人"把结果放进来
print("await Future →", await fut)


await Task → coro result
await Future → future result


In [15]:
# [3-4] 自定义可等待对象：实现 __await__（必须是一个生成器）
class CustomWaitable:
    """自定义可等待对象：内部委托给 asyncio.sleep 实现真实挂起"""
    def __init__(self, delay):
        self.delay = delay
    def __await__(self):
        # ① __await__ 必须返回生成器；可用 yield from 委托给其他可等待对象
        yield from asyncio.sleep(self.delay).__await__()
        return f"custom waitable done ({self.delay}s)"

r = await CustomWaitable(0.1)
print("await 自定义可等待对象 →", r)


await 自定义可等待对象 → custom waitable done (0.1s)


### 3.5 反例剧场

**易错点①：忘记 `await`** —— 协程创建了但从未执行，拿到的也不是结果。
**易错点②：在同步函数中写 `await`** —— 直接是语法错误（编译期就拦下）。


In [16]:
# [反例3-1] 忘记 await：拿到的是协程对象，不是结果
async def fetch(url):
    await asyncio.sleep(0.05)
    return f"data from {url}"

c = fetch("url-1")                  # ① 反例：忘记 await，只创建了协程
print("拿到的类型 →", type(c))       # ② <class 'coroutine'>，不是 str！
result = await c                     # ③ 补救：await 它才真正执行
print("await 之后 →", result)

# [反例3-2] 同步函数中写 await：SyntaxError（编译期报错）
src = "def sync_fn():\n    await fetch('x')\n"
try:
    compile(src, "<demo>", "exec")   # 编译即检查语法
except SyntaxError as e:
    print("同步函数中 await → SyntaxError:", e.msg)


拿到的类型 → <class 'coroutine'>


await 之后 → data from url-1
同步函数中 await → SyntaxError: 'await' outside async function


---

## 4 asyncio 核心 API

**本章解决什么问题**：把日常异步开发最高频的 API 一次讲透 —— 什么时候用 `gather`、什么时候用 `create_task`、超时怎么加、任务怎么保护。每个 API 都配可运行验证。

### API 速查表

| API | 一句话职责 | 详见 |
|---|---|---|
| `asyncio.run` | 脚本入口：建循环 → 跑 → 关 | 4.1 |
| `asyncio.sleep` | 非阻塞等待，让出控制权 | 4.2 |
| `asyncio.gather` | 并发一组可等待对象，按传入顺序返回结果 | 4.3 |
| `asyncio.create_task` | 把协程包装成 Task 立即调度 | 4.4 |
| `asyncio.wait` | 等一组任务，可控制返回条件 | 4.5 |
| `asyncio.as_completed` | 谁先完成先处理谁 | 4.6 |
| `asyncio.wait_for` | 给单个可等待对象加超时 | 4.7 |
| `asyncio.shield` | 保护任务不被外层取消波及 | 4.8 |
| `current_task` / `all_tasks` | 观察当前/全部任务 | 4.9 |
| `asyncio.TaskGroup` | 3.11+ 结构化并发，异常处理更安全 | 4.10 |


### 4.1 `asyncio.run` 与 4.2 `asyncio.sleep`

`asyncio.run(main())` 是**脚本**的入口函数（Notebook 内核自带循环，改用顶层 `await`，见 0.4）。`asyncio.sleep` 与 `time.sleep` 的本质区别：前者**让出控制权**，后者**霸占线程**。


In [17]:
# [4-2] asyncio.sleep 的非阻塞验证：5 个 sleep(0.2) 并发，总耗时仍 ≈ 0.2s
import asyncio, time

t0 = time.perf_counter()
await asyncio.gather(*[asyncio.sleep(0.2) for _ in range(5)])
t = time.perf_counter() - t0
print("5 × sleep(0.2) 并发 → 总耗时 < 0.35（≈ 单次 0.2）:", t < 0.35)
# 对照：5 次 time.sleep(0.2) 串行将是 ≈ 1.0s —— 这就是"让出控制权"的意义


5 × sleep(0.2) 并发 → 总耗时 < 0.35（≈ 单次 0.2）: True


### 4.3 `asyncio.gather`：并发 + 结果有序

并发运行多个可等待对象，**结果按传入顺序返回**（与完成先后无关）。


In [18]:
# [4-3a] gather：结果顺序 = 传入顺序（不是完成顺序）
async def llm(prompt, delay):
    await asyncio.sleep(delay)        # 模拟不同快慢的 LLM 调用
    return f"[{prompt}·耗时{delay}s]"

results = await asyncio.gather(
    llm("慢请求", 0.3),                # ① 后完成
    llm("快请求", 0.1),                # ② 先完成
)
print("结果顺序与【传入顺序】一致:", results)


结果顺序与【传入顺序】一致: ['[慢请求·耗时0.3s]', '[快请求·耗时0.1s]']


> **【修正说明 ①】** 源文档称："如果其中一个抛异常，默认会取消其他任务并抛出"。**实测结论**：`gather`（默认 `return_exceptions=False`）在某任务抛异常时会**立即向 await 方抛出该异常**，但**不会取消其他任务** —— 其余任务会继续在后台执行完毕。需要"一损俱损"的取消语义时，应使用 `TaskGroup`（4.10）或手动 `cancel`。下面用输出验证：

| 参数 | 行为 |
|---|---|
| `return_exceptions=False`（默认） | 首个异常立即抛给 await 方；**其余任务继续运行** |
| `return_exceptions=True` | 异常不抛出，作为结果列表中的元素返回；所有任务跑完 |


In [19]:
# [4-3b] 验证修正①：一个任务抛异常后，其他任务【没有被取消】
async def ok_task():
    await asyncio.sleep(0.2)
    print("  [ok_task] 异常已抛给 await 方，但我仍在运行 → 证明 gather 没有取消我")
    return "ok"

async def boom_task():
    await asyncio.sleep(0.05)
    raise ValueError("LLM API 500")

try:
    await asyncio.gather(ok_task(), boom_task())
except ValueError as e:
    print("① gather 立即向 await 方抛出 →", e)
    await asyncio.sleep(0.25)         # ② 留时间让 ok_task 跑完（它还活着）
    print("③ 结论：其余任务不会被自动取消（要取消请用 TaskGroup / 手动 cancel）")

# return_exceptions=True：异常作为结果返回，不打断其他任务
results = await asyncio.gather(ok_task(), boom_task(), return_exceptions=True)
print("④ return_exceptions=True →", results)


① gather 立即向 await 方抛出 → LLM API 500


  [ok_task] 异常已抛给 await 方，但我仍在运行 → 证明 gather 没有取消我


③ 结论：其余任务不会被自动取消（要取消请用 TaskGroup / 手动 cancel）


  [ok_task] 异常已抛给 await 方，但我仍在运行 → 证明 gather 没有取消我
④ return_exceptions=True → ['ok', ValueError('LLM API 500')]


### 4.4 `asyncio.create_task`：立即调度

`create_task` 把协程包装成 Task 并**立刻注册进事件循环** —— 不需要等到 `await` 才开始跑。`await task` 只是"等它的结果"，不是"启动它"。


In [20]:
# [4-4] create_task：创建即调度；await 只是等结果
async def job(name, delay):
    print(f"  [{name}] 被调度执行")
    await asyncio.sleep(delay)
    return f"{name} 完成"

t1 = asyncio.create_task(job("T1", 0.3))    # ① 创建即进入事件循环就绪队列
t2 = asyncio.create_task(job("T2", 0.1))
print("两个任务已创建，尚未 await 它们")       # ② 本行先打印（任务要等控制权让出才开始）
r1 = await t1                                # ③ await 只是等待结果
r2 = await t2
print("结果:", r1, "/", r2)


两个任务已创建，尚未 await 它们
  [T1] 被调度执行
  [T2] 被调度执行


结果: T1 完成 / T2 完成


### 4.5 `asyncio.wait`：控制返回条件

> **【修正说明 ②】** 源文档示例直接把协程列表传给 `asyncio.wait`。自 Python 3.11 起**已移除**对裸协程的支持（3.8 起弃用）—— **必须先包装成 `Task`**。本示例按正确写法执行。

`return_when` 可选值：`FIRST_COMPLETED`（任一完成即返回）/ `FIRST_EXCEPTION`（首个异常即返回）/ `ALL_COMPLETED`（全部完成，默认）。


In [21]:
# [4-5] asyncio.wait：FIRST_COMPLETED + 超时，正确写法（传 Task 而非协程）
async def job(name, delay):
    await asyncio.sleep(delay)
    return f"{name} 完成"

tasks = {
    asyncio.create_task(job("A", 0.3), name="job-A"),
    asyncio.create_task(job("B", 0.1), name="job-B"),
}
done, pending = await asyncio.wait(tasks, timeout=1.0, return_when=asyncio.FIRST_COMPLETED)

print("最先完成的 →", [t.result() for t in done])            # B（0.1s 先完成）
print("仍在运行的 →", sorted(t.get_name() for t in pending))  # job-A
for t in pending:                                            # 收尾：取消并回收，避免任务悬挂
    t.cancel()
await asyncio.gather(*pending, return_exceptions=True)
print("已清理未完成任务（无悬挂警告）")


最先完成的 → ['B 完成']
仍在运行的 → ['job-A']
已清理未完成任务（无悬挂警告）


### 4.6 `asyncio.as_completed`：按完成顺序处理

适合"谁先回来先处理谁"的流式场景（如渐进式渲染 LLM 回答）。完成顺序取决于各任务耗时。


In [22]:
# [4-6] as_completed：完成顺序 = 延迟从小到大（0.1 → 0.2 → 0.3）
async def job(name, delay):
    await asyncio.sleep(delay)
    return f"{name} 完成"

order = []
for fut in asyncio.as_completed([job("A", 0.3), job("B", 0.1), job("C", 0.2)]):
    r = await fut
    order.append(r)
    print("先完成先处理 →", r)
print("完成顺序:", order)


先完成先处理 → B 完成


先完成先处理 → C 完成
先完成先处理 → A 完成
完成顺序: ['B 完成', 'C 完成', 'A 完成']


### 4.7 `asyncio.wait_for`：超时控制

> **【修正说明 ③】** 自 Python 3.11 起，`asyncio.TimeoutError` **就是内建 `TimeoutError` 的别名**（同一个类）。捕获时写 `except TimeoutError` 即可，无需再区分两者。

超时后 `wait_for` 会**取消**内部的任务并抛出 `TimeoutError`。


In [23]:
# [4-7] wait_for：超时 → 取消内部任务并抛 TimeoutError
async def slow_llm():
    await asyncio.sleep(1.0)          # 模拟一次必然超时的慢调用
    return "结果"

print("验证 asyncio.TimeoutError is TimeoutError →", asyncio.TimeoutError is TimeoutError)

try:
    r = await asyncio.wait_for(slow_llm(), timeout=0.2)
except TimeoutError:
    print("0.2s 超时 → 内部任务已被取消，可走降级逻辑（返回兜底结果 / 重试）")


验证 asyncio.TimeoutError is TimeoutError → True


0.2s 超时 → 内部任务已被取消，可走降级逻辑（返回兜底结果 / 重试）

### 4.8 `asyncio.shield`：保护任务不被取消

场景：外层加了超时，但内部是**必须完成的 critical 操作**（如写库、计费落盘）。`shield` 让"外层等不及"不会连带取消内部任务。


In [24]:
# [4-8] shield：外层超时，但关键写操作继续执行完毕
async def critical_write():
    await asyncio.sleep(0.3)          # 模拟写向量库 / 计费落盘
    print("  [critical_write] 写操作完成（外层已超时，但我被 shield 保护）")
    return "written"

task = asyncio.create_task(critical_write())
try:
    await asyncio.wait_for(asyncio.shield(task), timeout=0.1)   # ① 外层只等 0.1s
except TimeoutError:
    print("① 外层等待超时 —— 但任务未被取消")
    result = await task               # ② 继续等它真正落盘
    print("② 最终拿到结果 →", result)


① 外层等待超时 —— 但任务未被取消


  [critical_write] 写操作完成（外层已超时，但我被 shield 保护）


② 最终拿到结果 → written


### 4.9 `asyncio.current_task` / `all_tasks`

`current_task()` 返回当前正在执行的任务；`all_tasks()` 返回该循环中所有**未完成**的任务。给任务命名（`name=`）能让调试输出一目了然。


In [25]:
# [4-9] current_task / all_tasks：给任务命名，观察运行中的任务
async def worker(tag):
    print(f"  [{tag}] 我所在的任务名 = {asyncio.current_task().get_name()}")
    await asyncio.sleep(0.2)
    return tag

tasks = [asyncio.create_task(worker(f"W{i}"), name=f"W{i}") for i in range(3)]
await asyncio.sleep(0.05)             # 让 3 个任务都跑到挂起点
running = sorted(t.get_name() for t in asyncio.all_tasks() if t.get_name().startswith("W"))
print("挂起中的任务:", running)        # ['W0', 'W1', 'W2']
print("顶层 await 也在某个任务里执行:", asyncio.current_task() is not None)
await asyncio.gather(*tasks)
print("全部结束（结束后 all_tasks 不再包含它们）")


  [W0] 我所在的任务名 = W0
  [W1] 我所在的任务名 = W1
  [W2] 我所在的任务名 = W2
挂起中的任务: ['W0', 'W1', 'W2']
顶层 await 也在某个任务里执行: True


全部结束（结束后 all_tasks 不再包含它们）


### 4.10 `asyncio.TaskGroup`（Python 3.11+）：结构化并发

`async with TaskGroup()` 退出时**自动等待所有子任务**；任何一个子任务抛异常时，**其余子任务会被自动取消**，异常以 `ExceptionGroup` 抛出（用 `except*` 捕获）。比 `gather` 更安全，是 3.11+ 的首选。


In [26]:
# [4-10a] TaskGroup：退出时自动等待全部子任务
async def job(name, delay):
    await asyncio.sleep(delay)
    print(f"  [{name}] 完成")
    return name

async def main():
    async with asyncio.TaskGroup() as tg:
        tg.create_task(job("A", 0.3), name="A")
        tg.create_task(job("B", 0.1), name="B")
    # 走到这里时，A、B 都已结束 —— 这就是"结构化并发"：任务的生命周期被限制在块内
    print("TaskGroup 退出：A、B 都已结束")

await main()


  [B] 完成


  [A] 完成


TaskGroup 退出：A、B 都已结束


In [27]:
# [4-10b] TaskGroup 异常语义：一错全停（对照 4-3b 的 gather：一错其他照跑）
async def fail_fast():
    await asyncio.sleep(0.05)
    raise RuntimeError("LLM 网关 500")

async def slow_one():
    try:
        await asyncio.sleep(1.0)
    except asyncio.CancelledError:
        print("  [slow_one] 被自动取消 —— 这正是 TaskGroup 比 gather 更安全的地方")
        raise                          # 捕获 CancelledError 后必须重新抛出（见 11 章）

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(fail_fast())
        tg.create_task(slow_one(), name="slow")
except* RuntimeError as eg:            # 3.11+ 异常组语法
    print("捕获 ExceptionGroup →", eg.exceptions[0])


  [slow_one] 被自动取消 —— 这正是 TaskGroup 比 gather 更安全的地方
捕获 ExceptionGroup → LLM 网关 500


### 4.11 反例剧场

**易错点①：串行 `await`** —— 连写两个 `await` 是先后等待，不是并发，性能直接退化到同步版本。
**易错点②：`create_task` 后不保存引用** —— 事件循环对任务只持**弱引用**，任务可能被垃圾回收中途蒸发（官方文档明确警告）。


In [28]:
# [反例4-1] 串行 await vs gather：同样的任务，耗时差 3 倍
async def job(name, delay):
    await asyncio.sleep(delay)
    return name

t0 = time.perf_counter()
await job("a", 0.1); await job("b", 0.1); await job("c", 0.1)   # ① 反例：串行等待
t_serial = time.perf_counter() - t0

t0 = time.perf_counter()
await asyncio.gather(job("a", 0.1), job("b", 0.1), job("c", 0.1))  # ② 正确：gather 并发
t_par = time.perf_counter() - t0

print(f"串行 ≈ {round(t_serial, 1)}s，并发 ≈ {round(t_par, 1)}s")
print("验证：串行 > 2.5 × 并发 →", t_serial > 2.5 * t_par)


串行 ≈ 0.3s，并发 ≈ 0.1s
验证：串行 > 2.5 × 并发 → True


In [29]:
# [反例4-2] create_task 后不保存引用：任务变成失控的 fire-and-forget
async def heartbeat():
    await asyncio.sleep(0.1)
    print("  [heartbeat] 跑完了")
    return "beat"

# ① 反例：创建后不保存任何引用 —— 任务照样被调度（fire-and-forget）
asyncio.create_task(heartbeat())
# ①-1 此刻手里没有任何句柄：无法 await 它、无法 cancel 它、拿不到结果/异常
strays = [t for t in asyncio.all_tasks() if t.get_coro().__name__ == "heartbeat"]
print("① 任务在后台运行，只能靠 all_tasks 瞥到它一眼:", len(strays) == 1)
await asyncio.sleep(0.15)             # ①-2 它自己跑完了
print("① 跑完即消失：返回值无处领取、异常无人知晓（官方还警告：弱引用下任务可能被 GC 中途回收）")

# ② 正确：保存引用 —— 可等待、可取消、可拿结果
saved = set()
t2 = asyncio.create_task(heartbeat())
saved.add(t2)
print("② 保存引用后能正常拿到结果 →", await t2)
saved.discard(t2)


① 任务在后台运行，只能靠 all_tasks 瞥到它一眼: True
  [heartbeat] 跑完了
① 跑完即消失：返回值无处领取、异常无人知晓（官方还警告：弱引用下任务可能被 GC 中途回收）


  [heartbeat] 跑完了
② 保存引用后能正常拿到结果 → beat


---

## 5 并发模式

**本章解决什么问题**：真实业务里"能不能跑起来"只是及格线，重点是**限流、流水线、超时重试、取消**这些工程化模式。全部围绕 LLM 调用场景展开。

### 模式总览

| 模式 | 核心工具 | 适用场景 |
|---|---|---|
| 5.1 gather 并发 | `gather` | 任务数量已知、都要结果 |
| 5.2 任务列表 | 推导式 + `create_task` | 从数据源动态生成任务 |
| 5.3 信号量限流 | `Semaphore` | 保护下游服务（LLM 配额） |
| 5.4 生产者消费者 | `Queue` | 流式产生 / 多 worker 消费 |
| 5.5 超时重试 | `wait_for` + 循环 | 应对瞬时故障 |
| 5.6 并发取消 | `cancel` + `CancelledError` | 任务生命周期管理 |

### 5.1 / 5.2 gather 并发与任务列表


In [30]:
# [5-1] gather 并发：批量调用 LLM（模拟）
async def llm_complete(prompt, delay):
    await asyncio.sleep(delay)        # 模拟网络延迟
    return f"[{prompt} 的回答]"

results = await asyncio.gather(
    llm_complete("翻译任务", 0.2),
    llm_complete("摘要任务", 0.1),
    llm_complete("改写任务", 0.15),
)
print("按传入顺序返回:", results)


按传入顺序返回: ['[翻译任务 的回答]', '[摘要任务 的回答]', '[改写任务 的回答]']


In [31]:
# [5-2] 任务列表：从数据源动态生成并发任务
prompts = [f"prompt-{i}" for i in range(5)]
tasks = [asyncio.create_task(llm_complete(p, 0.05)) for p in prompts]   # ① 列表推导批量建任务
results = await asyncio.gather(*tasks)                                   # ② * 展开并发
print("批量完成数量:", len(results))
print("结果:", results)


批量完成数量: 5
结果: ['[prompt-0 的回答]', '[prompt-1 的回答]', '[prompt-2 的回答]', '[prompt-3 的回答]', '[prompt-4 的回答]']


### 5.3 信号量限流

并发不是越大越好：一次打 1000 个请求会直接打爆 LLM 网关配额。`Semaphore(n)` 把同时在跑的任务数压在 n 以内 —— 注意它限制的是**同时在临界区内**的数量，不是总任务数。


In [32]:
# [5-3] Semaphore(2) 限流：6 个任务、峰值并发不超过 2
stats = {"active": 0, "peak": 0}
sem = asyncio.Semaphore(2)

async def llm_call(i):
    async with sem:                   # ① 同一时刻最多 2 个任务能进入
        stats["active"] += 1
        stats["peak"] = max(stats["peak"], stats["active"])
        await asyncio.sleep(0.1)      # ② 模拟调用 LLM
        stats["active"] -= 1
    return i

done = await asyncio.gather(*[llm_call(i) for i in range(6)])
print("完成任务数:", len(done), "；实测峰值并发 =", stats["peak"])
print("验证：峰值 ≤ 限流值 2 →", stats["peak"] <= 2)


完成任务数: 6 ；实测峰值并发 = 2
验证：峰值 ≤ 限流值 2 → True


### 5.4 生产者消费者

`asyncio.Queue` 是单线程内的任务流水线核心：生产者往里放，多个消费者从里取。用**哨兵值（None）**通知消费者退出，比源文档"消费者死循环 + 外部 cancel"的写法更可控（消费者能做完手头工作再退出）。


In [33]:
# [5-4] 生产者消费者：3 个消费者处理动态产生的任务
async def producer(queue, n_items, n_consumers):
    for i in range(n_items):
        await asyncio.sleep(0.02)     # ① 模拟任务产生（如接收评测请求）
        await queue.put(i)
        print(f"[生产] 任务 {i} 入队")
    for _ in range(n_consumers):
        await queue.put(None)         # ② 哨兵：每个消费者收到一个 None 后退出

async def consumer(queue, cid):
    processed = []
    while True:
        item = await queue.get()
        if item is None:              # ③ 收到哨兵：做完退出
            print(f"[消费者{cid}] 收到哨兵，共处理 {len(processed)} 个: {processed}")
            break
        await asyncio.sleep(0.05)     # ④ 模拟消费（如调用 LLM）
        processed.append(item)

async def main():
    q = asyncio.Queue()
    consumers = [asyncio.create_task(consumer(q, i)) for i in range(3)]
    await producer(q, 6, 3)
    await asyncio.gather(*consumers)  # ⑤ 等消费者自然退出（而不是粗暴 cancel）
    print("全部处理完成")

await main()


[生产] 任务 0 入队
[生产] 任务 1 入队
[生产] 任务 2 入队
[生产] 任务 3 入队
[生产] 任务 4 入队
[生产] 任务 5 入队
[消费者0] 收到哨兵，共处理 2 个: [0, 3]
[消费者1] 收到哨兵，共处理 2 个: [1, 4]
[消费者2] 收到哨兵，共处理 2 个: [2, 5]
全部处理完成


### 5.5 超时与重试

对外的每一次网络调用都应有三件套：**超时（wait_for）+ 重试（带上限）+ 确定性故障模拟**。下面的 `flaky_llm` 模拟"前 2 次超时、第 3 次成功"的瞬时故障，输出可复现。


In [34]:
# [5-5] 超时 + 带上限的重试
call_count = {}

async def flaky_llm(prompt, fail_times=2):
    call_count[prompt] = call_count.get(prompt, 0) + 1
    if call_count[prompt] <= fail_times:
        await asyncio.sleep(1.0)      # ① 模拟卡死（必然触发 0.2s 超时）
    await asyncio.sleep(0.02)
    return f"回答: {prompt}"

async def fetch_with_retry(prompt, retries=3, timeout=0.2):
    for i in range(retries):          # ② 重试必须带上限，否则雪崩
        try:
            return await asyncio.wait_for(flaky_llm(prompt), timeout)
        except TimeoutError:
            print(f"  [重试] {prompt} 第 {i + 1} 次超时")
            if i == retries - 1:
                raise                 # ③ 用尽重试仍失败 → 抛出，交给上层降级
    # 不可达

print("最终结果 →", await fetch_with_retry("翻译任务#1"))
print("实际调用次数:", call_count["翻译任务#1"], "（2 次失败 + 1 次成功）")


  [重试] 翻译任务#1 第 1 次超时


  [重试] 翻译任务#1 第 2 次超时
最终结果 → 回答: 翻译任务#1
实际调用次数: 3 （2 次失败 + 1 次成功）


### 5.6 并发取消

`task.cancel()` 会在任务**当前挂起点**抛出 `CancelledError`。任务内部应捕获它做清理，然后**必须重新抛出**（详见第 11 章陷阱 6）。


In [35]:
# [5-6] 取消任务：内层清理后 re-raise，外层观察到取消
async def long_running():
    try:
        await asyncio.sleep(5)        # 挂起点：cancel 的异常从这里抛入
    except asyncio.CancelledError:
        print("① [任务内部] 捕获取消 → 清理资源 → 重新抛出")
        raise

task = asyncio.create_task(long_running())
await asyncio.sleep(0.05)             # 让任务先跑到挂起点
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("② [外部] 观察到任务被取消")
print("任务最终状态 cancelled() →", task.cancelled())


① [任务内部] 捕获取消 → 清理资源 → 重新抛出
② [外部] 观察到任务被取消
任务最终状态 cancelled() → True


---

## 6 同步原语

**本章解决什么问题**：单线程无锁 ≠ 永远不需要同步。只要存在 `await` 挂起点，"检查 → 写入"就可能被其他任务插队。本章验证竞态如何发生，以及 `Lock / Event / Condition / Semaphore / Queue` 各自的适用场景。

### threading 与 asyncio 同步原语对照

| 原语 | threading 版 | asyncio 版 | 典型用途 |
|---|---|---|---|
| 互斥锁 | `Lock` | `asyncio.Lock`（用 `async with`） | 保护跨 await 的临界区 |
| 事件 | `Event` | `asyncio.Event` | 广播"就绪/关闭"信号 |
| 条件变量 | `Condition` | `asyncio.Condition` | "有数据了再叫醒我" |
| 信号量 | `Semaphore` | `asyncio.Semaphore` | 限流（见 5.3） |
| 队列 | `queue.Queue` | `asyncio.Queue` | 生产者消费者（见 5.4） |

> 关键差异：asyncio 版原语**不可跨线程使用**，且必须 `await` 获取（`async with lock`）。

### 6.1 Lock：验证"单线程也会竞态"


In [36]:
# [6-1] 无锁竞态 vs 加锁：await 挂起点打断了"读-改-写"
async def run_updates(use_lock):
    counter = [0]                     # 闭包内可变的计数器
    lock = asyncio.Lock()

    async def unsafe():
        tmp = counter[0]              # ① 读
        await asyncio.sleep(0)        # ② 挂起点：其他任务插进来，counter 还是旧值
        counter[0] = tmp + 1          # ③ 写：覆盖丢失

    async def safe():
        async with lock:              # 临界区内"读-改-写"不被插队
            tmp = counter[0]
            await asyncio.sleep(0)
            counter[0] = tmp + 1

    coros = [(safe() if use_lock else unsafe()) for _ in range(20)]
    await asyncio.gather(*coros)
    return counter[0]

print("不加锁 →", await run_updates(False), "（20 次自增只剩 1，丢失更新）")
print("加锁   →", await run_updates(True), "（结果正确）")


不加锁 → 1 （20 次自增只剩 1，丢失更新）
加锁   → 20 （结果正确）


### 6.2 Event：一次性广播"就绪"

`Event` 适合"一个信号、多个等待者"：`await event.wait()` 挂起，`event.set()` 后**所有**等待者同时恢复。


In [37]:
# [6-2] Event：服务就绪后，所有调用方同时开工
started = asyncio.Event()

async def service():
    print("① 服务初始化（加载模型…）")
    await asyncio.sleep(0.2)
    print("③ 服务就绪 → set 事件")
    started.set()                     # ④ 所有 wait() 的任务同时恢复

async def caller(i):
    await started.wait()              # ② 先到的调用方在此挂起等待
    print(f"④ 调用方 {i} 在就绪后开始工作")

async def main():
    callers = [asyncio.create_task(caller(i)) for i in range(2)]
    await asyncio.create_task(service())
    await asyncio.gather(*callers)

await main()


① 服务初始化（加载模型…）


③ 服务就绪 → set 事件


④ 调用方 0 在就绪后开始工作
④ 调用方 1 在就绪后开始工作


### 6.3 Condition：精确唤醒

比 Event 更细粒度：配合共享状态检查（`while not 条件: await cond.wait()`），`notify()` 唤醒一个、`notify_all()` 唤醒全部。


In [38]:
# [6-3] Condition：队列空 → 消费者挂起；生产者放入数据并 notify
cond = asyncio.Condition()
shared = []

async def consumer_c():
    async with cond:
        while not shared:             # ① 必须用 while（被唤醒后重新检查条件，防虚假唤醒）
            print("① [消费者] 队列为空，wait 挂起")
            await cond.wait()
        print(f"③ [消费者] 取走 {shared.pop()}")

async def producer_c():
    await asyncio.sleep(0.1)
    async with cond:
        shared.append("任务#1")
        print("② [生产者] 放入数据并 notify")
        cond.notify()                 # 唤醒一个等待者

await asyncio.gather(consumer_c(), producer_c())


① [消费者] 队列为空，wait 挂起
② [生产者] 放入数据并 notify
③ [消费者] 取走 任务#1


[None, None]

### 6.4 / 6.5 Semaphore 与 Queue

`Semaphore` 的限流用法已在 5.3 验证（峰值并发实测 ≤ 限流值）。`Queue` 的核心 API：

| 操作 | 语义 |
|---|---|
| `await queue.put(x)` | 放入；**队列满时挂起**（背压） |
| `await queue.get()` | 取出；队列空时挂起 |
| `queue.task_done()` | 通知 join：取出的这条已处理完 |
| `await queue.join()` | 等所有放入的元素都被 task_done |


In [39]:
# [6-4] 有界 Queue 的背压：队列满时生产者自动挂起
q = asyncio.Queue(maxsize=2)

async def putter():
    for i in range(4):
        await q.put(i)                # 队列满（size=2）时在此挂起，等消费者腾位置
        print(f"[put] 放入 {i}，当前大小 {q.qsize()}")
    print("[put] 全部放入")

async def getter():
    while True:
        item = await q.get()
        print(f"[get] 取走 {item}")
        q.task_done()                 # 每取一条都要 task_done，join 才能返回
        await asyncio.sleep(0.05)     # 模拟处理耗时
        if item == 3:
            return

async def main():
    p = asyncio.create_task(putter())
    g = asyncio.create_task(getter())
    await q.join()                    # 等 4 个元素全部 task_done
    await asyncio.gather(p, g)

await main()
print("背压验证：put 从未超过 maxsize=2 堆积")


[put] 放入 0，当前大小 1
[put] 放入 1，当前大小 2
[get] 取走 0
[put] 放入 2，当前大小 2
[get] 取走 1
[put] 放入 3，当前大小 2
[put] 全部放入


[get] 取走 2
[get] 取走 3


背压验证：put 从未超过 maxsize=2 堆积


---

## 7 异步生成器与推导式

**本章解决什么问题**：LLM 的流式输出（SSE / chunk）本质就是异步生成器；`async with` 则是管理连接、会话等异步资源的标准姿势。

### 7.1 异步生成器

`async def` + `yield` = 异步生成器：每 `yield` 一个值前可以 `await`（等网络数据到达）。


In [40]:
# [7-1] 异步生成器：模拟 LLM 流式输出
async def stream_llm(prompt, n_chunks=3):
    for i in range(n_chunks):
        await asyncio.sleep(0.05)     # ① 模拟等待下一个 chunk 从网络到达
        yield f"[{prompt}·chunk{i}]"  # ② 吐出一块，暂停在这里

async def main():
    async for chunk in stream_llm("你好"):   # ③ async for 逐块消费
        print("收到 →", chunk)

await main()


收到 → [你好·chunk0]


收到 → [你好·chunk1]
收到 → [你好·chunk2]


### 7.2 异步推导式

列表 / 字典 / 集合推导都支持 `async for`（须在异步上下文中使用）。


In [41]:
# [7-2] 异步推导式：列表推导与字典推导
async def numbered(agen):
    # 给异步生成器的产出编号，便于做字典推导
    i = 0
    async for c in agen:
        yield i, c
        i += 1

async def main():
    chunks = [c async for c in stream_llm("摘要")]           # ① 异步列表推导
    print("① 列表推导:", chunks)
    d = {i: c async for i, c in numbered(stream_llm("翻译"))}  # ② 异步字典推导
    print("② 字典推导:", d)

await main()


① 列表推导: ['[摘要·chunk0]', '[摘要·chunk1]', '[摘要·chunk2]']
② 字典推导: {0: '[翻译·chunk0]', 1: '[翻译·chunk1]', 2: '[翻译·chunk2]'}


### 7.3 异步上下文管理器

两种写法：类实现 `__aenter__` / `__aexit__`，或用 `@asynccontextmanager` 装饰生成器函数（写法更轻）。


In [42]:
# [7-3] 异步上下文管理器的两种写法
from contextlib import asynccontextmanager

# 写法①：类实现 __aenter__ / __aexit__
class AsyncDBConn:
    async def __aenter__(self):
        print("① 建立数据库连接")
        return self
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("③ 关闭连接（无论是否异常都执行）")
        return False
    async def query(self, sql):
        await asyncio.sleep(0.05)     # 模拟查询
        return f"rows of {sql}"

async def main_class():
    async with AsyncDBConn() as conn:
        print("②", await conn.query("SELECT 1"))

await main_class()

# 写法②：@asynccontextmanager（适合简单资源）
@asynccontextmanager
async def llm_session(model):
    print(f"① 加载模型 {model}")
    try:
        yield model                    # yield 的值 = as 绑定的值
    finally:
        print("③ 释放模型资源")

async def main_dec():
    async with llm_session("qwen-7b") as m:
        print(f"② 使用 {m} 推理")

await main_dec()


① 建立数据库连接


② rows of SELECT 1
③ 关闭连接（无论是否异常都执行）
① 加载模型 qwen-7b
② 使用 qwen-7b 推理
③ 释放模型资源


### 7.4 异步迭代器

异步生成器是语法糖，底层协议是 `__aiter__` + `__anext__`。自定义异步迭代器适合"每次取下一个都要等 I/O"的场景。


In [43]:
# [7-4] 手写异步迭代器：AsyncCounter
class AsyncCounter:
    def __init__(self, n):
        self.n = n                     # 上限
        self.i = 0                     # 当前值
    def __aiter__(self):
        return self                    # 返回实现了 __anext__ 的对象
    async def __anext__(self):
        if self.i >= self.n:
            raise StopAsyncIteration   # 终止信号
        self.i += 1
        await asyncio.sleep(0.02)      # 模拟异步获取下一个元素
        return self.i

async def main():
    async for x in AsyncCounter(3):
        print("迭代到 →", x)

await main()


迭代到 → 1
迭代到 → 2
迭代到 → 3


### 7.5 反例剧场

**易错点：异步生成器是一次性的** —— 消费完再迭代得到空序列（不报错，但拿不到数据）；提前放弃消费时应 `aclose()` 触发清理。


In [44]:
# [反例7-1] 生成器耗尽后再次消费 → 空列表（静默失败）
agen = stream_llm("测试", 2)
print("第一次消费 →", [c async for c in agen])
print("第二次消费 →", [c async for c in agen], "（耗尽，静默返回空！要新数据需重新调用 stream_llm()）")

# [反例7-2] 提前放弃消费：应显式 aclose() 触发 finally 清理
async def agen_with_cleanup():
    try:
        yield 1
        yield 2
    finally:
        print("② aclose() 触发了清理逻辑")

g = agen_with_cleanup()
print("① 取第一个值 →", await g.__anext__())
await g.aclose()                       # ③ 不再消费就显式关闭
print("③ 已显式关闭")


第一次消费 → ['[测试·chunk0]', '[测试·chunk1]']
第二次消费 → [] （耗尽，静默返回空！要新数据需重新调用 stream_llm()）
① 取第一个值 → 1
② aclose() 触发了清理逻辑
③ 已显式关闭


---

## 8 实用示例集（工程高频模式）

**本章解决什么问题**：把前面的知识点组装成 5+1 个可直接抄走用的模式。每个模式说明适用场景与注意点。

| # | 模式 | 核心组合 | 适用场景 | 注意点 |
|---|---|---|---|---|
| ① | 批量调用器 | Semaphore + wait_for + 重试 | 批量调用 LLM / 第三方 API | 重试必须有上限 |
| ② | 先到先处理 | as_completed | 流式渲染、渐进式展示 | 结果顺序 ≠ 完成顺序 |
| ③ | 关键操作保护 | shield + wait_for | 计费、落盘等不可半途而废的写 | 外层超时 ≠ 任务失败 |
| ④ | 单飞缓存 | Future 字典 | 防缓存击穿（同 key 并发只放行一个回源） | Future 要记得清理 |
| ⑤ | 优雅关闭 | Event + 后台任务循环 | 服务收到 SIGTERM 后有序退出 | 信号注册不能在 Jupyter 里做，见注 |
| ⑥ | 子进程管理 | create_subprocess_exec | 调用外部命令且不阻塞循环 | 用 communicate() 收输出 |


### 模式①：并发 + 限流 + 超时 + 重试 的批量 LLM 调用器


In [45]:
# [模式①] LLMBatchRunner：四件套齐全的批量调用器（全部确定性模拟）
import asyncio, time

class LLMBatchRunner:
    def __init__(self, concurrency=3, timeout=0.15, retries=2):
        self.sem = asyncio.Semaphore(concurrency)   # ① 限流：最多 3 路并发
        self.timeout = timeout                      # ② 单次调用超时
        self.retries = retries                      # ③ 重试上限
        self.attempts = 0

    async def _call(self, prompt_id):
        # 模拟 LLM API：按 prompt_id 确定性地产出延迟 / 故障
        self.attempts += 1
        if prompt_id == 4:              # 4 号 prompt 永久故障（怎么重试都超时）
            await asyncio.sleep(1.0)
        await asyncio.sleep(0.03 + (prompt_id % 3) * 0.03)   # 正常延迟 0.03~0.09s

    async def run_one(self, prompt_id):
        for attempt in range(1, self.retries + 1):
            async with self.sem:        # ④ 限流作用域包住整个网络调用
                try:
                    await asyncio.wait_for(self._call(prompt_id), self.timeout)
                    return prompt_id, "ok"
                except TimeoutError:
                    if attempt == self.retries:
                        return prompt_id, "failed"   # 用尽重试 → 标记失败（也可 raise）
        return prompt_id, "unreachable"

    async def run(self, n):
        t0 = time.perf_counter()
        results = await asyncio.gather(*[self.run_one(i) for i in range(n)])
        return results, time.perf_counter() - t0

runner = LLMBatchRunner()
results, dt = await runner.run(8)
print("结果:", results)
print("成功:", sum(1 for _, s in results if s == "ok"), "/ 8（4 号 prompt 永久故障）")
print("总尝试次数（含重试）:", runner.attempts, "→ 7×1 + 2 = 9")
print("耗时远小于串行下限（9 次尝试 × 0.15s = 1.35s+）:", dt < 1.0)


结果: [(0, 'ok'), (1, 'ok'), (2, 'ok'), (3, 'ok'), (4, 'failed'), (5, 'ok'), (6, 'ok'), (7, 'ok')]
成功: 7 / 8（4 号 prompt 永久故障）
总尝试次数（含重试）: 9 → 7×1 + 2 = 9
耗时远小于串行下限（9 次尝试 × 0.15s = 1.35s+）: True


### 模式②：先到先处理（流式聚合）

适合"结果到一个展示一个"的场景：不等最慢的那个，谁先回来先处理谁。


In [46]:
# [模式②] as_completed：谁先完成先处理（模拟 5 路 LLM 调用）
async def llm(prompt_id):
    await asyncio.sleep(0.05 + ((prompt_id * 2) % 5) * 0.05)   # 延迟互不相同：0.05/0.15/0.25/0.10/0.20
    return f"回答{prompt_id}"

order = []
for fut in asyncio.as_completed([llm(i) for i in range(5)]):
    r = await fut
    order.append(r)
    print("先到先处理 →", r)
print("完成顺序（延迟互不相同时完全确定）:", order)


先到先处理 → 回答0
先到先处理 → 回答3
先到先处理 → 回答1


先到先处理 → 回答4
先到先处理 → 回答2
完成顺序（延迟互不相同时完全确定）: ['回答0', '回答3', '回答1', '回答4', '回答2']


### 模式③：`shield` 保护关键写操作

外层的超时控制用于"不让调用方干等"，但内部的写操作（计费、落盘）必须完成 —— 两者解耦。


In [47]:
# [模式③] 外层 0.1s 超时返回，内部写操作 0.3s 后照常落盘
async def submit_batch(rows):
    await asyncio.sleep(0.3)           # 模拟提交到向量库 / 计费系统
    return f"已写入 {len(rows)} 条"

async def main():
    task = asyncio.create_task(submit_batch(["r1", "r2"]))
    try:
        return await asyncio.wait_for(asyncio.shield(task), timeout=0.1)
    except TimeoutError:
        print("① 外层 0.1s 超时返回（调用方不等了），但写入未取消…")
        return await task              # 服务侧继续等它真正落盘

print("②", await main())


① 外层 0.1s 超时返回（调用方不等了），但写入未取消…


② 已写入 2 条


### 模式④：单飞缓存（single-flight），防缓存击穿

并发多个请求要同一个 key：第一个请求回源加载，其余请求**合并等待同一个 Future** —— 回源只发生一次。这是对源文档 8.8 节"锁 + 二次检查"写法的改进（原写法在锁释放到写入之间，多个请求仍会重复回源）。


In [48]:
# [模式④] 单飞缓存：同 key 并发 5 个请求，只回源 1 次
cache = {}        # key → 数据
inflight = {}     # key → Future（正在回源中的"占位符"）
load_count = [0]

async def loader(key):
    load_count[0] += 1
    await asyncio.sleep(0.1)           # 模拟昂贵的回源（查库 / 调 LLM）
    return f"data({key})"

async def get_data(key):
    if key in cache:                                   # ① 命中缓存
        return cache[key], "cache"
    if key in inflight:                                # ② 已有请求在回源 → 合并等待
        return await inflight[key], "merged"
    fut = asyncio.get_running_loop().create_future()   # ③ 我来回源，登记 Future
    inflight[key] = fut
    try:
        data = await loader(key)
        cache[key] = data
        fut.set_result(data)                           # ④ 唤醒所有合并等待者
        return data, "loaded"
    finally:
        inflight.pop(key, None)                        # ⑤ 无论成败都清理登记

results = await asyncio.gather(*[get_data("user:1") for _ in range(5)])
print("5 个并发请求的结果来源:", results)
print("实际回源次数:", load_count[0], "→ 只加载 1 次，其余 4 个请求合并等待")


5 个并发请求的结果来源: [('data(user:1)', 'loaded'), ('data(user:1)', 'merged'), ('data(user:1)', 'merged'), ('data(user:1)', 'merged'), ('data(user:1)', 'merged')]
实际回源次数: 1 → 只加载 1 次，其余 4 个请求合并等待


### 模式⑤：优雅关闭

真实服务里用 `loop.add_signal_handler(signal.SIGTERM, handler)` 注册信号（Jupyter 内核占用信号机制，不能在 Notebook 里演示注册，故用任务模拟"收到停机信号"）；要点是：**后台任务循环检查 Event，而不是被外部粗暴 cancel**。


In [49]:
# [模式⑤] 优雅关闭：停机信号 → 各 worker 完成清理后退出
stop = asyncio.Event()                # 真实服务中由信号处理器调用 stop.set()

async def worker_loop(name):
    while not stop.is_set():           # ① 每轮检查停机信号
        await asyncio.sleep(0.05)      # ② 模拟处理任务
    print(f"  [{name}] 收到停机信号，完成清理后退出")   # ③ 有机会做清理

async def shutdown_after(t):
    await asyncio.sleep(t)
    stop.set()                         # 模拟 SIGTERM 到达

async def main():
    workers = [asyncio.create_task(worker_loop(n)) for n in ("推理worker", "日志worker")]
    await asyncio.create_task(shutdown_after(0.15))
    await asyncio.gather(*workers)
    print("服务优雅关闭（所有 worker 均有序退出）")

await main()


  [推理worker] 收到停机信号，完成清理后退出
  [日志worker] 收到停机信号，完成清理后退出
服务优雅关闭（所有 worker 均有序退出）


### 模式⑥：子进程管理（可真实运行）

调用外部命令且不阻塞事件循环。


In [50]:
# [模式⑥] 子进程：echo 的输出真实可验证、完全确定
proc = await asyncio.create_subprocess_exec(
    "echo", "hello-from-subprocess",   # ① 要执行的外部命令
    stdout=asyncio.subprocess.PIPE,    # ② 捕获标准输出
    stderr=asyncio.subprocess.PIPE,
)
stdout, _ = await proc.communicate()  # ③ 等待结束并收全输出
print("子进程输出 →", stdout.decode().strip())
print("退出码 →", proc.returncode)


子进程输出 → hello-from-subprocess
退出码 → 0


### 真实生态代码参考（依赖第三方库，不在本 Notebook 执行）

```python
# ① 并发 HTTP：aiohttp
import aiohttp

async def fetch(session, url):
    async with session.get(url) as resp:
        return await resp.text()

async def main():
    urls = [f"https://example.com/{i}" for i in range(10)]
    async with aiohttp.ClientSession() as session:      # 会话复用连接池
        tasks = [fetch(session, u) for u in urls]
        results = await asyncio.gather(*tasks)

# ② 爬虫限流：aiohttp + Semaphore
sem = asyncio.Semaphore(5)                              # 同时最多 5 路请求

async def fetch(session, url):
    async with sem:
        async with session.get(url) as resp:
            return await resp.text()

# ③ 数据库批量操作：asyncpg（PostgreSQL）
import asyncpg

async def main():
    conn = await asyncpg.connect("postgresql://...")
    try:
        await conn.executemany(
            "INSERT INTO users(name) VALUES($1)",
            [("Alice",), ("Bob",), ("Carol",)],
        )
    finally:
        await conn.close()

# ④ Web 框架：FastAPI（handler 本身就是协程）
from fastapi import FastAPI
import httpx

app = FastAPI()

@app.get("/users")
async def get_users():
    async with httpx.AsyncClient() as client:
        resp = await client.get("https://api.example.com/users")
        return resp.json()

# ⑤ 异步文件读写：aiofiles
import aiofiles

async def main():
    async with aiofiles.open("data.txt", "w") as f:
        await f.write("hello")
```


---

## 9 内置 / 生态对照

**本章解决什么问题**：选型看两张表 —— 同步原语怎么对应、同步库换成哪个异步库。

### 9.1 同步库 → 异步库对照

| 领域 | 同步方案 | 异步方案 | 备注 |
|---|---|---|---|
| HTTP 客户端 | `requests` | `aiohttp` / `httpx.AsyncClient` | httpx 同步异步双模 |
| 等待 | `time.sleep` | `asyncio.sleep` | 最常犯的错（见陷阱 3） |
| 文件 | `open` | `aiofiles` | |
| PostgreSQL | `psycopg2` | `asyncpg` | |
| Redis | `redis-py`（同步模式） | `redis.asyncio` | 同一个包 |
| 消息队列 | `pika` | `aio-pika` | |
| CPU 密集 | —— | `loop.run_in_executor` + `ProcessPoolExecutor` | 异步里跑 CPU 任务的正确姿势 |

### 9.2 三种并发模型对比（源文档第 10 章）

| 维度 | 异步 | 多线程 | 多进程 |
|---|---|---|---|
| 并发模型 | 协作式 | 抢占式 | 并行 |
| 适用场景 | I/O 密集 | I/O 密集 | CPU 密集 |
| GIL 影响 | 无 | 有 | 无 |
| 共享状态 | 单线程无锁 | 需要锁 | 需要 IPC |
| 开销 | 最小 | 中 | 大 |
| 调试难度 | 较高 | 中 | 高 |
| 生态支持 | 部分库不支持 | 全支持 | 全支持 |

**选择建议**：I/O 密集 + 高并发 → 异步；已有同步库、不想改 → 多线程；CPU 密集 → 多进程；混合负载 → 异步 + `run_in_executor`。


---

## 10 综合实战：LLM 批量评测流水线

**本章解决什么问题**：把限流、超时、重试、TaskGroup、Queue 五个知识点组装成一个贴近 LLMOps 业务的完整服务。

### 业务需求

评测服务要对 **20 个评测 prompt** 逐个调用 LLM 并统计成功率：

1. **限流**：下游 LLM 网关配额有限，并发不得超过 4
2. **超时**：单次调用 0.15s 未返回即视为失败
3. **重试**：失败最多重试 1 次（共 2 次尝试）
4. **故障注入**（确定性模拟）：`pid % 7 == 3` 的调用**永久故障**；`pid % 5 == 2` 的调用**第 1 次超时、重试即成功**
5. **统计**：结果经 `Queue` 汇给聚合器，输出成功/失败/重试次数

### 设计取舍（先看结论再看代码）

| 决策 | 选择 | 理由 |
|---|---|---|
| 并发控制 | `Semaphore(4)` 而非分批 | 分批（chunk）会造成批尾空闲；信号量让任务"完成一个补一个"，吞吐更高 |
| 超时实现 | `wait_for` 包住调用 | 在调用方加超时，不侵入 API 实现；超时后任务被干净取消 |
| 任务编排 | `TaskGroup` 而非 `gather` | 任何子任务抛异常时自动取消兄弟任务、异常以 ExceptionGroup 汇总，不泄漏任务 |
| 结果汇总 | `Queue` + 聚合器任务 | 统计逻辑与执行逻辑解耦；换成写库 / 推送只需替换消费者 |
| 失败策略 | 重试用尽后标记 failed 而非 raise | 评测场景要的是统计报告，单条失败不应中断整批 |


In [51]:
# [10-1] 综合实战：LLM 批量评测流水线（全部确定性模拟，输出可复现）
import asyncio, time

CONCURRENCY, TIMEOUT, MAX_RETRY, N_PROMPTS = 4, 0.15, 2, 20

attempt_log = []                        # (pid, 第几次尝试)
result_queue = asyncio.Queue()          # 只传递最终结果
peak_active = [0]
sem = asyncio.Semaphore(CONCURRENCY)
active = [0]

class FakeLLMAPI:
    # 模拟 LLM 服务：按 pid 确定性地决定延迟与故障类型
    def __init__(self):
        self.calls = 0
    async def complete(self, pid, attempt):
        self.calls += 1
        if pid % 7 == 3:                            # ① 永久故障：每次都拖到超时
            await asyncio.sleep(1.0)
        if attempt == 1 and pid % 5 == 2:           # ② 瞬时故障：第 1 次超时，重试即好
            await asyncio.sleep(1.0)
        await asyncio.sleep(0.03 + (pid % 4) * 0.02)  # ③ 正常延迟 0.03~0.09s

async def eval_one(api, pid):
    """评测单个 prompt：限流 → 超时 → 重试，最终把结果交给聚合器"""
    for attempt in range(1, MAX_RETRY + 1):
        attempt_log.append((pid, attempt))
        async with sem:                             # ① 限流：最多 4 路并发
            active[0] += 1
            peak_active[0] = max(peak_active[0], active[0])
            try:
                await asyncio.wait_for(api.complete(pid, attempt), TIMEOUT)  # ② 超时
                status = "ok"
            except TimeoutError:
                status = "timeout"                  # ③ 稍后重试或标记失败
            finally:
                active[0] -= 1
        if status == "ok":
            await result_queue.put((pid, "ok"))
            return
    await result_queue.put((pid, "failed"))         # ④ 用尽重试 → 失败（不 raise，不中断整批）

async def aggregator(n):
    """聚合器：从队列收满 n 条最终结果后返回统计"""
    stats = {"ok": 0, "failed": 0}
    while sum(stats.values()) < n:
        pid, status = await result_queue.get()
        stats[status] += 1
    return stats

async def main():
    api = FakeLLMAPI()
    t0 = time.perf_counter()
    async with asyncio.TaskGroup() as tg:           # ⑤ TaskGroup：结构化编排
        agg = tg.create_task(aggregator(N_PROMPTS))
        for pid in range(N_PROMPTS):
            tg.create_task(eval_one(api, pid))
    dt = time.perf_counter() - t0
    return agg.result(), api.calls, dt

stats, calls, dt = await main()
retries = calls - N_PROMPTS
print("成功 / 失败:", stats["ok"], "/", stats["failed"], "（永久故障 pid: 3, 10, 17）")
print("API 总调用（含重试）:", calls, "；其中重试:", retries, "次")
print("实测峰值并发:", peak_active[0], "≤ 限流值", CONCURRENCY, "→", peak_active[0] <= CONCURRENCY)
print("总耗时远小于串行下限（26 次尝试 × 0.15s = 3.9s）:", dt < 1.5)


成功 / 失败: 17 / 3 （永久故障 pid: 3, 10, 17）
API 总调用（含重试）: 26 ；其中重试: 6 次
实测峰值并发: 4 ≤ 限流值 4 → True
总耗时远小于串行下限（26 次尝试 × 0.15s = 3.9s）: True


### 运行结果解读

- **成功 17 / 失败 3**：pid 3、10、17 永久故障（`pid % 7 == 3`），重试用尽后标记 failed；pid 2、7、12 瞬时故障（`pid % 5 == 2`）第 2 次尝试成功 —— 这解释了"总调用 26 次 = 14×1 + 6×2"
- **峰值并发 = 4**：信号量全程压住了并发上限，保护了下游配额
- **总耗时 ≈ 0.7s**：远低于串行下限 3.9s —— 收益来自两处重叠：超时等待之间互相重叠 + 重试与其他任务重叠

### 若要投入生产，还需补齐

1. **真实客户端**：`FakeLLMAPI` 换成 `httpx.AsyncClient` / SDK，超时与连接池交给客户端配置
2. **退避策略**：重试间加指数退避（`await asyncio.sleep(0.1 * 2 ** attempt)`），避免雪崩
3. **可观测**：`attempt_log` 换成结构化日志 / metrics 上报
4. **幂等**：重试的前提是下游接口幂等，否则需要请求级去重（可结合模式④的缓存）


---

## 11 常见陷阱（陷阱 / 症状 / 解法）

**本章解决什么问题**：把 12 类高频陷阱汇成一张表；未在前文反例剧场覆盖的，这里逐一给出可运行的复现代码（报错均被 try/except 捕获展示，保证单元 0 报错）。

| # | 陷阱 | 症状 | 解法 | 复现位置 |
|---|---|---|---|---|
| 1 | 忘记 `await` | 拿到 `<coroutine object>`；协程从未执行；RuntimeWarning | 检查返回类型；开 `-W error::RuntimeWarning` | 3.5 ✅ |
| 2 | 同步函数中 `await` | SyntaxError（编译期） | 包进 `async def`，入口 `asyncio.run` | 3.5 ✅ |
| 3 | 阻塞事件循环 | 一个 `time.sleep` 拖慢所有并发任务 | 换 `asyncio.sleep`；阻塞调用丢 `run_in_executor` | 11-1 ✅ |
| 4 | 串行 `await` | 耗时 = 各任务之和 | `gather` / `TaskGroup` | 4.11 ✅ |
| 5 | 任务未保存引用 | 任务句柄丢失（fire-and-forget）：无法等待/取消/拿结果；官方警告任务可能被 GC 中途回收（"Task was destroyed but it is pending!"） | 存入集合，用完移除 | 4.11 ✅ |
| 6 | 吞掉 `CancelledError` | 取消"失效"，任务照常返回结果 | 捕获清理后必须 `raise` | 11-2 ✅ |
| 7 | `finally` 中 `await` | 清理中的 await 被二次取消打断 | 清理要快；关键清理用 `shield` | 11-3 ✅ |
| 8 | 混用多个事件循环 | "Event loop is closed" 等诡异错误 | 一个应用一个循环；跨循环对象禁止共享 | 11-4 ✅ |
| 9 | 异步生成器不关闭 | finally 清理不执行、ResourceWarning | 用完 `aclose()`（或 async with 上下文） | 7.5 ✅ |
| 10 | 跨线程直接操作 | asyncio 原语非线程安全，行为未定义 | 用 `run_coroutine_threadsafe` 提交 | 11-5 ✅ |
| 11 | 超时未处理 | `TimeoutError` 直接炸穿调用栈 | `wait_for` 外层 try/except + 降级 | 11-6 ✅ |
| 12 | `__init__` 中 `await` | SyntaxError | 异步工厂方法 `await cls.create()` | 11-7 ✅ |


In [52]:
# [11-1] 陷阱3：阻塞事件循环 —— 一个 time.sleep 拖慢所有任务
async def heartbeat():
    t0 = time.perf_counter()
    await asyncio.sleep(0.1)           # 本应 0.1s 后醒来
    return time.perf_counter() - t0

async def main_bad():
    task = asyncio.create_task(heartbeat())
    await asyncio.sleep(0.01)          # 让 heartbeat 挂起在 sleep 上
    time.sleep(0.3)                    # ① 反例：同步阻塞 —— 整个事件循环（含所有任务）卡死
    return await task

actual = await main_bad()
print(f"① 反例：heartbeat 预期 0.1s，实际等了 ≈ {round(actual, 1)}s →", actual > 0.25)

async def main_good():
    task = asyncio.create_task(heartbeat())
    await asyncio.sleep(0.01)
    loop = asyncio.get_running_loop()
    await loop.run_in_executor(None, time.sleep, 0.3)   # ② 正确：阻塞调用丢进线程池
    return await task

actual = await main_good()
print(f"② 修复：heartbeat 实际等了 ≈ {round(actual, 1)}s →", actual < 0.25)


① 反例：heartbeat 预期 0.1s，实际等了 ≈ 0.3s → True


② 修复：heartbeat 实际等了 ≈ 0.1s → True


In [53]:
# [11-2] 陷阱6：吞掉 CancelledError —— 取消"失效"
async def swallow_cancel():
    try:
        await asyncio.sleep(5)
    except asyncio.CancelledError:
        return "done"                  # ① 反例：捕获后不 re-raise，取消被吞

t = asyncio.create_task(swallow_cancel())
await asyncio.sleep(0.05)
t.cancel()
result = await t                       # ② 竟然能拿到返回值！
print("① 反例：被取消的任务返回了", repr(result), "；cancelled() =", t.cancelled())

async def proper_cancel():
    try:
        await asyncio.sleep(5)
    except asyncio.CancelledError:
        print("② 正确：清理资源…")
        raise                          # ③ 必须 re-raise

t2 = asyncio.create_task(proper_cancel())
await asyncio.sleep(0.05)
t2.cancel()
try:
    await t2
except asyncio.CancelledError:
    print("③ 外层正常观察到取消；cancelled() =", t2.cancelled())


① 反例：被取消的任务返回了 'done' ；cancelled() = False
② 正确：清理资源…
③ 外层正常观察到取消；cancelled() = True


In [54]:
# [11-3] 陷阱7：finally 中的 await 被二次取消打断
async def risky_cleanup():
    try:
        await asyncio.sleep(5)
    finally:
        print("① 进入 finally，开始异步清理（需要 0.2s）")
        try:
            await asyncio.sleep(0.2)   # 清理本身也要 await —— 危险点
        except asyncio.CancelledError:
            print("② 第二次 cancel 把清理中的 await 打断了！清理可能没做完")
            raise

t = asyncio.create_task(risky_cleanup())
await asyncio.sleep(0.05)
t.cancel()                             # 第一次取消：进入 finally
await asyncio.sleep(0.05)
if not t.done():
    t.cancel()                         # 第二次取消：打断清理中的 await
try:
    await t
except asyncio.CancelledError:
    print("③ 任务以取消结束")
# 解法：清理逻辑尽量同步完成；确需 await 的关键清理用 asyncio.shield 包住


① 进入 finally，开始异步清理（需要 0.2s）


② 第二次 cancel 把清理中的 await 打断了！清理可能没做完
③ 任务以取消结束


In [55]:
# [11-4] 陷阱8：混用多个事件循环 —— 跨循环对象失效
import threading

class Holder:
    pass

h = Holder()

def first():
    """循环①：创建 Event，让任务挂在它上面等待，然后关闭循环①"""
    loop = asyncio.new_event_loop()
    ev = asyncio.Event()

    async def waiter():
        await ev.wait()                # 挂起等待 —— 回调注册在循环①上

    h.task = loop.create_task(waiter())
    loop.run_until_complete(asyncio.sleep(0.05))   # 让 waiter 挂起
    h.ev = ev
    loop.close()                       # 循环① 关闭（waiter 仍挂着，保留引用防 GC 警告）

def second():
    """循环②：尝试 set 一个属于循环① 的 Event"""
    loop = asyncio.new_event_loop()
    async def use():
        h.ev.set()                     # 需要往循环①投递唤醒回调 → 循环①已关闭
    try:
        loop.run_until_complete(use())
    except RuntimeError as e:
        print("跨事件循环使用对象 →", e)
    finally:
        loop.close()

th1 = threading.Thread(target=first); th1.start(); th1.join()
th2 = threading.Thread(target=second); th2.start(); th2.join()
# 解法：整个应用只用一个事件循环（asyncio.run 只调一次）；
#       Notebook/框架环境里用内核自带的循环，不要另起炉灶


跨事件循环使用对象 → Event loop is closed


In [56]:
# [11-5] 陷阱10：跨线程操作 asyncio —— 正确姿势 run_coroutine_threadsafe
from concurrent.futures import ThreadPoolExecutor

async def work(x):
    await asyncio.sleep(0.05)
    return x * 2

loop = asyncio.get_running_loop()      # 主线程（内核）的循环
with ThreadPoolExecutor(1) as pool:
    # ① 在工作线程里把协程"安全提交"到主循环（返回 concurrent.futures.Future）
    cf = pool.submit(asyncio.run_coroutine_threadsafe, work(21), loop).result()
    # ② 主循环侧用 wrap_future 把它转成可 await 的 asyncio Future
    print("跨线程提交协程的结果 →", await asyncio.wrap_future(cf))
# 反例（不要写）：在工作线程里直接 loop.create_task(...) 或 await 事件循环原语 —— 非线程安全


跨线程提交协程的结果 → 42


In [57]:
# [11-6] 陷阱11：超时未处理 —— TimeoutError 炸穿调用栈
try:
    await asyncio.wait_for(asyncio.sleep(1.0), timeout=0.1)   # ① 反例：不捕获直接崩
except TimeoutError:
    print("① 捕获超时 → 走降级逻辑（返回兜底结果 / 记录 / 重试），调用栈不再炸")

# [11-7] 陷阱12：__init__ 中 await —— SyntaxError，正确姿势是异步工厂
src = "class A:\n    def __init__(self):\n        await self.init()\n"
try:
    compile(src, "<demo>", "exec")     # 编译期直接拦下
except SyntaxError as e:
    print("② __init__ 中 await → SyntaxError:", e.msg)

class LLMPool:
    def __init__(self):
        self.ready = False
    async def warmup(self):
        await asyncio.sleep(0.05)       # 模拟加载模型
        self.ready = True
        return self
    @classmethod
    async def create(cls):             # ③ 异步工厂方法
        self = cls()
        return await self.warmup()

pool = await LLMPool.create()
print("③ 工厂方法初始化完成，ready =", pool.ready)


① 捕获超时 → 走降级逻辑（返回兜底结果 / 记录 / 重试），调用栈不再炸
② __init__ 中 await → SyntaxError: 'await' outside async function


③ 工厂方法初始化完成，ready = True


---

## 12 总结

### 12.1 核心概念

| 概念 | 一句话 |
|---|---|
| 事件循环 | 调度中心：执行就绪任务、挂起等待者、I/O 就绪后唤醒 |
| 协程 | `async def` 定义的函数，调用返回协程对象、不执行 |
| `await` | 挂起当前协程等结果；期间事件循环去跑别的任务 |
| Task | 被事件循环调度的协程（`create_task` 的产物） |
| Future | 底层结果占位符："将来才有值" |

### 12.2 通用代码模板（可直接复制的骨架）

```python
import asyncio

# ---------- 1. 入口：脚本里只此一次 ----------
async def main():
    async with asyncio.TaskGroup() as tg:          # 2. 结构化并发（3.11+；旧版本用 gather）
        for item in items:
            tg.create_task(handle(item), name=f"task-{item}")

asyncio.run(main())                                # Notebook 里改为顶层 await main()

# ---------- 3. 带限流 / 超时 / 重试的单次调用 ----------
sem = asyncio.Semaphore(4)                         # 限流：保护下游

async def call_with_policy(coro_factory, timeout=5, retries=3):
    for attempt in range(1, retries + 1):
        async with sem:
            try:
                return await asyncio.wait_for(coro_factory(), timeout)
            except TimeoutError:
                if attempt == retries:
                    raise                          # 用尽重试，抛给上层降级
                await asyncio.sleep(0.1 * 2 ** (attempt - 1))   # 指数退避

# ---------- 4. 生产者消费者 ----------
async def producer(q: asyncio.Queue):
    for item in source:
        await q.put(item)                          # 有界队列自动背压
    await q.put(None)                              # 哨兵通知退出

async def consumer(q: asyncio.Queue):
    while (item := await q.get()) is not None:
        await process(item)

# ---------- 5. 资源管理 / 优雅关闭 ----------
from contextlib import asynccontextmanager

@asynccontextmanager
async def managed_resource():
    resource = await acquire()
    try:
        yield resource
    finally:
        await release(resource)                    # 异常也保证释放

stop = asyncio.Event()
async def serve():
    while not stop.is_set():
        await handle_one()
# 收到 SIGTERM 时：stop.set() → 各任务做完手头工作后退出
```

### 12.3 记忆口诀

1. **异步只救 I/O**，CPU 密集交给进程池（`run_in_executor`）
2. **入口 `asyncio.run`**，一个应用一个循环，Notebook 里顶层 `await`
3. **并发用 gather / TaskGroup**，连写 `await` 是串行大忌
4. **对外调用三件套**：Semaphore 限流、wait_for 超时、重试带上限
5. **create_task 必存引用**，CancelledError 捕获后必 `raise`
6. **循环之内禁阻塞**：`time.sleep` 是毒药，`asyncio.sleep` / `run_in_executor` 是解药


### 12.4 自测练习（先自己写，再看参考实现）

**练习 1**：实现 `fetch_all(urls, concurrency=2, timeout=0.5)` —— 并发抓取、限制并发、单个超时不中断整批（返回 `"timeout:{url}"` 占位），结果按输入顺序返回。

**练习 2**：服务依赖 DB 和 Cache 两个组件，用两个 `Event` 实现"两者都就绪才开始接收请求"。

**练习 3（改错）**：下面代码有 3 处异步错误，找出并改正：

```python
async def bad():
    task = asyncio.create_task(work())   # 错误③
    time.sleep(0.2)                      # 错误①
    r1 = await work()                    # 错误②
    r2 = await work()
    return [task, r1, r2]
```


In [58]:
# [练习1·参考实现] 并发 + 限流 + 超时 + 结果有序
async def fake_fetch(url):
    await asyncio.sleep(0.1)
    return f"data:{url}"

async def fetch_all(urls, concurrency=2, timeout=0.5):
    sem = asyncio.Semaphore(concurrency)             # ① 限流

    async def one(u):
        async with sem:
            try:
                return await asyncio.wait_for(fake_fetch(u), timeout)   # ② 超时
            except TimeoutError:
                return f"timeout:{u}"                # ③ 降级占位，不中断整批

    return await asyncio.gather(*[one(u) for u in urls])   # ④ 结果按输入顺序

urls = [f"u{i}" for i in range(6)]
print(await fetch_all(urls))


['data:u0', 'data:u1', 'data:u2', 'data:u3', 'data:u4', 'data:u5']


In [59]:
# [练习2·参考实现] 两个 Event 实现"依赖全部就绪才开工"
db_ready, cache_ready = asyncio.Event(), asyncio.Event()

async def wait_dependencies():
    # gather 两个 wait()：任意一个未就绪都会挂起，全部 set 后恢复
    await asyncio.gather(db_ready.wait(), cache_ready.wait())
    return "DB + Cache 全部就绪，开始接收请求"

async def boot():
    await asyncio.sleep(0.1); db_ready.set()         # 模拟 DB 初始化完成
    await asyncio.sleep(0.1); cache_ready.set()      # 模拟 Cache 初始化完成

t = asyncio.create_task(wait_dependencies())
await boot()
print(await t)


DB + Cache 全部就绪，开始接收请求


In [60]:
# [练习3·参考实现] 三处错误的修正
async def work():
    await asyncio.sleep(0.05)
    return "work done"

async def good():
    saved = set()                                    # 修正③：保存任务引用
    task = asyncio.create_task(work())
    saved.add(task)

    await asyncio.sleep(0.2)                         # 修正①：阻塞调用 → asyncio.sleep
    # （若必须调用阻塞函数：await loop.run_in_executor(None, time.sleep, 0.2)）

    r1, r2 = await asyncio.gather(work(), work())    # 修正②：串行 → gather 并发
    result = await task
    saved.discard(task)
    return [result, r1, r2]

print(await good())


['work done', 'work done', 'work done']


### 12.5 源文档修正说明汇总

| # | 源文档原文 | 修正后结论 | 验证位置 |
|---|---|---|---|
| ① | "gather 中一个抛异常，默认会取消其他任务" | 只向 await 方抛出异常，**其余任务继续运行**，不会被取消 | 4.3 输出证明 |
| ② | `asyncio.wait` 示例直接传协程列表 | 3.11 起移除裸协程支持，**必须包装为 Task** | 4.5 正确写法 |
| ③ | 超时捕获 `asyncio.TimeoutError` | 3.11 起它是内建 `TimeoutError` 的**别名**（同一对象） | 4.7 `is` 验证为 True |

另有两处**写法改进**（非错误）：5.4 生产者消费者改用哨兵退出（比"死循环 + 外部 cancel"更可控）；8.8 异步缓存改为单飞模式（原"锁 + 二次检查"在窗口期仍会重复回源）。

### 12.6 结语

> 异步的本质是"**单线程内协作式调度**"：遇到 I/O 就挂起，让事件循环去跑其他任务；它适合**高并发 I/O**，不适合 CPU 密集。

本 Notebook 全部代码单元已真实执行：**0 个报错单元**；所有涉及执行时机 / 顺序 / 等价性的论断均附验证输出；故障与延迟均为确定性模拟，输出可复现。
